[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GoogleCloudPlatform/conversational-insights-devkit/blob/main/examples/13-workflow-ops.ipynb)

# Colab Test Harness: Exporting & Importing Insights Configuration (`ConfigOps`)

This notebook lets you interactively test `cxidk.workflow.ops.ConfigOps` across
all supported Contact Center Insights configuration assets:

* **QA Scorecards** (latest revision + questions; customer-defined, Discovery Engine & OOTB)
* **Custom Question Tags** (`qaQuestionTags`)
* **Smart Highlighters** (`phraseMatchers`)
* **Topic Models** (`issueModels`)
* **Analysis Rules** (`analysisRules` — with cross-environment ID remapping)
* **Auto Labeling Rules** (`autoLabelingRules`)
* **Assessment Rules** (`assessmentRules`)
* **Correlation Rules** (`correlationConfig`)

You can run this notebook in two modes (selectable in **Step 1**):
1. **`LIVE_GCP`** — talks to your real Google Cloud project(s) and `gs://` bucket.
2. **`SANDBOX_DEMO`** — wires an in-memory Cloud Storage bucket and sample Insights
   resources so you can test-drive every cell in 5 seconds without touching GCP.

## 0. Install & Authenticate

In Google Colab this cell installs `cxidk` from the `main` branch and prompts
for Google Cloud authentication. In a local virtualenv it uses your local checkout
and Application Default Credentials.

In [ ]:
import importlib.util
import sys

if importlib.util.find_spec("cxidk") is None or not importlib.util.find_spec(
    "cxidk.workflow.ops"
):
    print("Installing cxidk from https://github.com/GoogleCloudPlatform/conversational-insights-devkit@main ...")
    !pip install -q "git+https://github.com/GoogleCloudPlatform/conversational-insights-devkit.git@main"

if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
    print("Authenticated via Google Colab.")
else:
    print("Using local Python environment and Application Default Credentials.")

from cxidk.workflow.ops import (
    Asset,
    ConfigOps,
    ExportOptions,
    ImportOptions,
    NamingPolicy,
    ReferenceResolution,
)

print("cxidk.workflow.ops imported successfully.")

## 1. Configure Environment & Mode

Fill in the form fields below and run the cell:

* Pick **`LIVE_GCP`** to test against real GCP projects and a real `gs://` bucket.
* Pick **`SANDBOX_DEMO`** to run against realistic in-memory fixtures first.

In [ ]:
# @title Environment & Bundle Settings { run: "auto" }
MODE = "LIVE_GCP"  # @param ["LIVE_GCP", "SANDBOX_DEMO"]
SOURCE_PROJECT = "insights-pipeline-producer"  # @param {type:"string"}
SOURCE_LOCATION = "us-central1"  # @param ["us-central1", "us-east1", "us-west1", "europe-west1", "europe-west2", "europe-west3", "asia-south1", "asia-southeast1", "asia-northeast1", "northamerica-northeast1", "australia-southeast1", "global"]
DEST_PROJECT = "insights-pipeline-producer"  # @param {type:"string"}
DEST_LOCATION = "us-central1"  # @param ["us-central1", "us-east1", "us-west1", "europe-west1", "europe-west2", "europe-west3", "asia-south1", "asia-southeast1", "asia-northeast1", "northamerica-northeast1", "australia-southeast1", "global"]
BUNDLE_URI = "gs://your-bucket-name/insights-export/bundle.json"  # @param {type:"string"}

if MODE == "SANDBOX_DEMO":
    import json
    from unittest.mock import MagicMock
    from cxidk.wrapper import insights as _insights
    from cxidk.wrapper import storage as _storage

    SOURCE_PROJECT = "sandbox-src-proj"
    DEST_PROJECT = "sandbox-dst-proj"
    BUNDLE_URI = "gs://demo-bundles/config.json"
    _SRC_PARENT = f"projects/{SOURCE_PROJECT}/locations/{SOURCE_LOCATION}"
    _DST_PARENT = f"projects/{DEST_PROJECT}/locations/{DEST_LOCATION}"
    _BLOB_STORE = {}

    class _DemoGcs:
        def __init__(self, bucket_name, **_kw):
            self.bucket_name = bucket_name

        def bucket_exists(self):
            return True

        def upload_blob(self, file_name, data, **_kw):
            _BLOB_STORE[(self.bucket_name, file_name)] = data

        def download_blob(self, file_name, **_kw):
            return _BLOB_STORE[(self.bucket_name, file_name)]

    _storage.Gcs = _DemoGcs

    def _stub_collection(src_items, collection_seg):
        dst_items = []

        class _Stub:
            def __init__(self, project_id, location, **_kw):
                self.project_id = project_id
                self.parent = f"projects/{project_id}/locations/{location}"

            def list(self, **_kw):
                return list(src_items if self.project_id == SOURCE_PROJECT else dst_items)

            def list_active(self, **_kw):
                return [r for r in self.list() if r.get("active", True)]

            def list_by_source(self, **_kw):
                return self.list()

            def list_deployed(self):
                return [r for r in self.list() if r.get("state") == "DEPLOYED"]

            def get_latest(self):
                items = self.list()
                return items[-1] if items else None

            def get(self):
                return {"fullConversationConfig": {"correlationRules": self.list()}}

            def update(self, body, **_kw):
                dst_items[:] = body["fullConversationConfig"]["correlationRules"]
                return body

            def create(self, body, resource_id=None):
                rid = resource_id or f"gen-{len(dst_items) + 1}"
                created = {**body, "name": f"{self.parent}/{collection_seg}/{rid}"}
                dst_items.append(created)
                return created

        return _Stub

    _insights.QaQuestionTags = _stub_collection(
        [{"name": f"{_SRC_PARENT}/qaQuestionTags/tag-1", "displayName": "Escalation"}],
        "qaQuestionTags",
    )
    _insights.QaScorecards = _stub_collection(
        [{
            "name": f"{_SRC_PARENT}/qaScorecards/sc-1",
            "displayName": "Core QA Scorecard",
            "source": "QA_SCORECARD_SOURCE_DISCOVERY_ENGINE",
            "isDefault": True,
        }],
        "qaScorecards",
    )
    _insights.ScorecardRevisions = _stub_collection(
        [{
            "name": f"{_SRC_PARENT}/qaScorecards/sc-1/revisions/rev-latest",
            "createTime": "2026-09-01T00:00:00Z",
        }],
        "revisions",
    )
    _insights.RevisionQuestions = _stub_collection(
        [{
            "name": f"{_SRC_PARENT}/qaScorecards/sc-1/revisions/rev-latest/qaQuestions/q-1",
            "abbreviation": "Proper greeting",
            "questionBody": "Did the agent greet the customer?",
            "tags": ["BUSINESS", f"{_SRC_PARENT}/qaQuestionTags/tag-1"],
            "tuningMetadata": {"totalValidLabelCount": 42},
        }],
        "qaQuestions",
    )
    _insights.PhraseMatchers = _stub_collection(
        [{
            "name": f"{_SRC_PARENT}/phraseMatchers/pm-1",
            "displayName": "Cancel request",
            "type": "ANY_OF",
            "active": True,
        }],
        "phraseMatchers",
    )
    _insights.IssueModels = _stub_collection([], "issueModels")
    _insights.AutoLabelingRules = _stub_collection([], "autoLabelingRules")
    _insights.AssessmentRules = _stub_collection([], "assessmentRules")
    _insights.CorrelationConfig = _stub_collection(
        [{"ruleId": "corr-1", "active": True, "joinKeyExpression": "conversation.customer_id"}],
        "correlationConfig",
    )
    _insights.AnalysisRules = _stub_collection(
        [{
            "name": f"{_SRC_PARENT}/analysisRules/ar-1",
            "displayName": "Primary Analysis Rule",
            "active": True,
            "analysisPercentage": 0.25,
            "annotatorSelector": {
                "phraseMatchers": [f"{_SRC_PARENT}/phraseMatchers/pm-1"],
                "qaConfig": {
                    "scorecardList": {
                        "qaScorecardRevisions": [
                            f"{_SRC_PARENT}/qaScorecards/sc-1/revisions/rev-latest"
                        ]
                    }
                },
            },
        }],
        "analysisRules",
    )
    print("SANDBOX_DEMO mode active: wired in-memory GCS + sample Insights resources.")

source = ConfigOps(SOURCE_PROJECT, SOURCE_LOCATION)
destination = ConfigOps(DEST_PROJECT, DEST_LOCATION)

print(
    f"Mode:        {MODE}\n"
    f"Source:      {source.project_id} / {source.location}\n"
    f"Destination: {destination.project_id} / {destination.location}\n"
    f"Bundle URI:  {BUNDLE_URI}"
)

## 1b. (Optional, `LIVE_GCP` only) Seed Sample Resources in `SOURCE_PROJECT`

If your `SOURCE_PROJECT` currently has no active resources, set `SEED_SAMPLE = True`
and run this cell to create one active phrase matcher and one scorecard in
`SOURCE_PROJECT` so the export has live data to pick up.

In [ ]:
SEED_SAMPLE = False  # @param {type:"boolean"}

if SEED_SAMPLE and MODE == "LIVE_GCP":
    from cxidk.wrapper import insights as wrapper_insights

    pm_client = wrapper_insights.PhraseMatchers(
        project_id=SOURCE_PROJECT, location=SOURCE_LOCATION
    )
    created_pm = pm_client.create(
        body={
            "displayName": "Ops Test Highlighter",
            "type": "ANY_OF",
            "active": True,
            "phraseMatchRuleGroups": [
                {
                    "type": "ANY_OF",
                    "phraseMatchRules": [
                        {
                            "query": "cancel my subscription",
                            "config": {"exactMatchConfig": {}},
                        }
                    ],
                }
            ],
        }
    )
    print("Created active PhraseMatcher:", created_pm.get("name"))
else:
    print("Skipped seeding (set SEED_SAMPLE = True in LIVE_GCP mode to create a test resource).")

## 2. Export Configuration to Cloud Storage

`export()` reads all active resources (or a subset via `assets=[...]`) and writes
a single JSON bundle to `BUNDLE_URI`.

In [ ]:
export_report = source.export(uri=BUNDLE_URI)

print(f"Exported {export_report.total} records -> {export_report.uri}\n")
for asset_name, outcome in export_report.assets.items():
    print(f"  {asset_name:<22} {outcome.count}")

### Inspect the Exported JSON Bundle

Let's read back the bundle from `BUNDLE_URI` to verify that:
1. Output-only fields (`name`, `createTime`, `updateTime`) were stripped.
2. Scorecard `tuningMetadata` / training data were excluded.
3. Discovery Engine / OOTB scorecard provenance is preserved in `_sourceEnum`.

In [ ]:
import json

bundle_doc = source.bundle_io.read_json(BUNDLE_URI)
print("=== MANIFEST ===")
print(json.dumps(bundle_doc["manifest"], indent=2))

print("\n=== NON-EMPTY ASSETS IN BUNDLE ===")
for key, val in bundle_doc["assets"].items():
    if val:
        print(f"\n--- {key} ---")
        print(json.dumps(val, indent=2)[:1200])

## 3. Preview the Import (`plan()`)

`plan()` is completely read-only: it checks for display-name clashes at the
destination, flags same-environment duplication, and translates
`analysisPercentage` (which the API stores in `[0, 1]`) into human-readable
billing notes.

In [ ]:
plan_report = destination.plan(
    uri=BUNDLE_URI,
    options=ImportOptions(activate_imported=False),
)
print(plan_report.summary())

## 4. Rehearse with `dry_run=True`

`dry_run=True` walks the full import & reference-rewriting pipeline without
performing any writes.

In [ ]:
dry_run_report = destination.import_(
    uri=BUNDLE_URI,
    options=ImportOptions(dry_run=True),
)

print(f"Dry run complete (dry_run={dry_run_report.dry_run})")
print(f"Would create {dry_run_report.total_created} resources across {len(dry_run_report.assets)} assets.")
for asset_name, outcome in dry_run_report.assets.items():
    if outcome.errors:
        print(f"  [ERROR] {asset_name}: {outcome.errors}")

## 5. Run the Create-Only Import (`import_()`)

Key invariants enforced during `import_()`:
* **Create-only**: never overwrites, merges, or deletes existing destination resources.
* **Fresh IDs**: every imported resource gets a new ID (`<stem>-<6hex>`) so
  running the import a second time creates another copy instead of failing with
  `ALREADY_EXISTS`.
* **Safe by default**: imported rules arrive with `active=False` and suffixed
  display names (`"{name} (imported YYYY-MM-DD)"`).
* **Reference remapping**: `AnalysisRule.annotatorSelector` and `QaQuestion.tags`
  are automatically rewritten from source resource names to the newly created
  destination resource names.

In [ ]:
import_report = destination.import_(
    uri=BUNDLE_URI,
    options=ImportOptions(
        naming=NamingPolicy.SUFFIX,
        activate_imported=False,
        reference_resolution=ReferenceResolution.STRICT,
    ),
)

print(f"Created {import_report.total_created} resources (failed={import_report.failed})\n")

for asset_name, outcome in import_report.assets.items():
    if outcome.count or outcome.warnings or outcome.errors:
        print(f"[{asset_name}] created={outcome.count} skipped={outcome.skipped}")
        for created_name in outcome.created_names:
            print(f"   + {created_name}")
        for warning in outcome.warnings:
            print(f"   ! warning: {warning}")
        for error in outcome.errors:
            print(f"   x error:   {error}")

print("\nNotes:")
for note in import_report.notes:
    print(f"  - {note}")

## 6. Export/Import a Specific Subset Only

You can also export or import individual asset types (for example, just
`PHRASE_MATCHERS` or just `SCORECARDS` + `QA_QUESTION_TAGS`).

In [ ]:
subset_report = destination.import_(
    uri=BUNDLE_URI,
    assets=[Asset.PHRASE_MATCHERS],
    options=ImportOptions(
        naming=NamingPolicy.DUPLICATE,  # keep exact display name
        activate_imported=False,
    ),
)

print(
    "Subset import (PHRASE_MATCHERS with NamingPolicy.DUPLICATE):",
    subset_report.assets["phrase_matchers"].created_names,
)